# ⚡ Pakistani Electric Meter Detector
### YOLOv8s trained on 11,000 images → EasyOCR reads digits

**Do this before running any cell:**
> Runtime → Change Runtime Type → **T4 GPU** → Save

Then run every cell top to bottom.

---

## Cell 1 — Check GPU

In [ ]:
!nvidia-smi
# Must show Tesla T4. If not: Runtime > Change Runtime Type > T4 GPU

## Cell 2 — Install Libraries

In [ ]:
!pip install ultralytics roboflow easyocr -q
print('✅ All libraries installed!')

## Cell 3 — Download 11k Dataset from Roboflow

In [ ]:
from roboflow import Roboflow
import os, yaml

rf      = Roboflow(api_key="0leDKPV6myoVoduqhsIy")
project = rf.workspace("aipoweredsmartmetermanagement").project("detector-electric-meter-otcse")
version = project.version(1)
dataset = version.download("yolov8")

# ── Fix data.yaml to absolute paths ─────────────────────────────────────────
DATA_YAML = os.path.join(dataset.location, 'data.yaml')

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(dataset.location, 'train', 'images')
data['val']   = os.path.join(dataset.location, 'valid', 'images')
test_dir      = os.path.join(dataset.location, 'test',  'images')
if os.path.exists(test_dir):
    data['test'] = test_dir

with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

# ── Report ───────────────────────────────────────────────────────────────────
train_n = len(os.listdir(data['train']))
val_n   = len(os.listdir(data['val']))

print(f"\n✅ Dataset ready!")
print(f"   Train  : {train_n} images")
print(f"   Val    : {val_n} images")
print(f"   Classes: {data['nc']} → {data['names']}")
print(f"   YAML   : {DATA_YAML}")

## Cell 4 — Preview Annotations (verify labels are correct)

In [ ]:
import cv2, os, random
import matplotlib.pyplot as plt

img_dir = data['train']
lbl_dir = img_dir.replace('images', 'labels')
all_imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
sample   = random.sample(all_imgs, min(6, len(all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, fname in enumerate(sample):
    img  = cv2.imread(os.path.join(img_dir, fname))
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl  = os.path.join(lbl_dir, os.path.splitext(fname)[0] + '.txt')

    if os.path.exists(lbl):
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, cx, cy, bw, bh = map(float, parts)
                    x1 = int((cx-bw/2)*w); y1 = int((cy-bh/2)*h)
                    x2 = int((cx+bw/2)*w); y2 = int((cy+bh/2)*h)
                    cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 3)
                    name = data['names'][int(cls)] if int(cls)<len(data['names']) else str(int(cls))
                    cv2.putText(img, name, (x1, max(y1-8,0)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,0,0), 2)

    axes[idx].imshow(img)
    axes[idx].set_title(fname, fontsize=8)
    axes[idx].axis('off')

plt.suptitle('Annotations — GREEN box must be around meter display', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ If boxes look correct → run Cell 5')
print('❌ If boxes look wrong  → fix labels on Roboflow first')

## Cell 5 — Train YOLOv8s
> 11k images × 100 epochs ≈ **60–90 minutes** on T4 GPU
>
> Do NOT close the browser tab. Keep Colab tab open.

In [ ]:
from ultralytics import YOLO

# yolov8s = small model, best balance of speed vs accuracy for this dataset size
model = YOLO('yolov8s.pt')

model.train(
    data      = DATA_YAML,
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,       # T4 has 15GB — 16 fits comfortably
    patience  = 20,       # stops early if no improvement for 20 epochs
    device    = 0,
    workers   = 4,
    name      = 'meter_v3',
    exist_ok  = True,

    # Light augmentation — 11k images is plenty, don't overdo it
    hsv_h     = 0.015,
    hsv_s     = 0.5,
    hsv_v     = 0.4,
    degrees   = 10.0,
    translate = 0.1,
    scale     = 0.3,
    fliplr    = 0.5,
    mosaic    = 1.0,
)

print('\n✅ Training complete!')
print('Best weights → /content/runs/detect/meter_v3/weights/best.pt')

## Cell 6 — View Training Results

In [ ]:
from IPython.display import Image as IPImage
import os

base = '/content/runs/detect/meter_v3'

for fpath, title in [
    (f'{base}/results.png',              'Training Curves'),
    (f'{base}/confusion_matrix.png',     'Confusion Matrix'),
    (f'{base}/val_batch0_pred.jpg',      'Sample Predictions'),
]:
    if os.path.exists(fpath):
        print(f'\n📊 {title}:')
        display(IPImage(fpath, width=900))

## Cell 7 — Validate & Get Real mAP Score

In [ ]:
from ultralytics import YOLO

BEST_PT   = '/content/runs/detect/meter_v3/weights/best.pt'
model     = YOLO(BEST_PT)
metrics   = model.val(data=DATA_YAML, device=0)

map50  = metrics.box.map50
map5095= metrics.box.map
prec   = metrics.box.mp
rec    = metrics.box.mr

print('\n' + '='*45)
print('  VALIDATION RESULTS')
print('='*45)
print(f'  mAP@0.50      : {map50:.4f}')
print(f'  mAP@0.50:0.95 : {map5095:.4f}')
print(f'  Precision     : {prec:.4f}')
print(f'  Recall        : {rec:.4f}')
print('='*45)

if map50 >= 0.85:
    print('\n✅ EXCELLENT — model is production ready!')
elif map50 >= 0.70:
    print('\n✅ GOOD — will work well for meter detection')
elif map50 >= 0.50:
    print('\n⚠️  DECENT — functional but could be better')
else:
    print('\n❌ LOW — check annotations quality on Roboflow')

## Cell 8 — Test Detection on Your Photo

In [ ]:
from google.colab import files
from ultralytics import YOLO
from IPython.display import Image as IPImage
import glob

BEST_PT = '/content/runs/detect/meter_v3/weights/best.pt'
model   = YOLO(BEST_PT)

print('📸 Upload a meter photo:')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

results = model(img_path, conf=0.25, save=True,
                project='/content/test_out',
                name='pred', exist_ok=True)

saved = glob.glob('/content/test_out/pred/*.jpg') + \
        glob.glob('/content/test_out/pred/*.png')
if saved:
    display(IPImage(saved[0], width=700))

boxes = results[0].boxes
if boxes and len(boxes) > 0:
    print(f'\n✅ Meter detected! ({len(boxes)} box(es))')
    for i, b in enumerate(boxes):
        x1,y1,x2,y2 = map(int, b.xyxy[0].tolist())
        print(f'   Box {i+1}: ({x1},{y1})→({x2},{y2})  conf: {float(b.conf[0]):.2f}')
else:
    print('❌ Not detected at conf=0.25 — trying 0.1...')
    results2 = model(img_path, conf=0.10, verbose=False)
    boxes2   = results2[0].boxes
    if boxes2 and len(boxes2) > 0:
        print(f'✅ Found at conf=0.10 — use conf=0.1 in Cell 9')
    else:
        print('❌ Still nothing. Paste this output and ask for help.')

## Cell 9 — Full Pipeline: YOLO crops meter → EasyOCR reads digits

In [ ]:
import easyocr, cv2, numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from google.colab import files

BEST_PT  = '/content/runs/detect/meter_v3/weights/best.pt'
detector = YOLO(BEST_PT)
reader   = easyocr.Reader(['en'], gpu=True)
print('✅ YOLO + EasyOCR loaded!')

# ─────────────────────────────────────────────────────────────────────────────
def read_meter(image_path, conf=0.25):
    img     = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w    = img.shape[:2]

    # ── Phase 1: YOLO ────────────────────────────────────────────────────────
    res   = detector(image_path, conf=conf, verbose=False)
    boxes = res[0].boxes

    if boxes and len(boxes) > 0:
        best         = boxes[boxes.conf.argmax()]
        x1,y1,x2,y2 = map(int, best.xyxy[0].tolist())
        pad          = 15
        x1,y1        = max(0,x1-pad), max(0,y1-pad)
        x2,y2        = min(w,x2+pad), min(h,y2+pad)
        crop         = img_rgb[y1:y2, x1:x2]
        detected     = True
        cscore       = float(best.conf[0])
    else:
        # fallback: use full image
        crop         = img_rgb
        x1,y1,x2,y2 = 0, 0, w, h
        detected     = False
        cscore       = 0.0

    # ── Phase 2: Preprocess crop ─────────────────────────────────────────────
    gray     = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    upscaled = cv2.resize(gray, (gray.shape[1]*3, gray.shape[0]*3),
                          interpolation=cv2.INTER_CUBIC)
    denoised = cv2.fastNlMeansDenoising(upscaled, h=10)
    kernel   = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
    sharp    = cv2.filter2D(denoised, -1, kernel)

    # ── Phase 3: EasyOCR ─────────────────────────────────────────────────────
    raw     = reader.readtext(sharp, allowlist='0123456789.',
                               detail=1, paragraph=False, min_size=10)
    raw     = sorted(raw, key=lambda x: x[0][0][0])          # left → right
    digits  = [(t.strip(), round(c,2)) for _,t,c in raw
                if t.strip() and c > 0.25]
    reading = ''.join([d[0] for d in digits])

    # ── Visualise ────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    overlay = img_rgb.copy()
    color   = (0,255,0) if detected else (255,165,0)
    cv2.rectangle(overlay, (x1,y1), (x2,y2), color, 4)
    label   = f'Meter  conf:{cscore:.2f}' if detected else 'Fallback: full image'
    cv2.putText(overlay, label, (x1, max(y1-12,0)),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3)
    axes[0].imshow(overlay)
    axes[0].set_title('Phase 1 — YOLO Detection', fontsize=13, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(sharp, cmap='gray')
    axes[1].set_title('Phase 2 — Preprocessed Crop', fontsize=13, fontweight='bold')
    axes[1].axis('off')

    axes[2].imshow(crop)
    axes[2].set_title(f'Phase 3 — Reading: {reading}',
                      fontsize=14, fontweight='bold',
                      color='green' if reading else 'red')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    print('\n' + '='*42)
    print(f'  METER READING : {reading if reading else "NOT FOUND"}')
    print('='*42)
    if digits:
        for txt, c in digits:
            print(f'    "{txt}"   conf: {c}')

    return reading
# ─────────────────────────────────────────────────────────────────────────────

print('\n📸 Upload a meter image:')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]
read_meter(img_path)

## Cell 10 — Save Model to Google Drive

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/meter_model_v3'
os.makedirs(save_dir, exist_ok=True)

for fname in ['best.pt', 'last.pt']:
    src = f'/content/runs/detect/meter_v3/weights/{fname}'
    dst = f'{save_dir}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✅ Saved: {dst}')

print(f'\nUse best.pt for inference.')

---
## Quick Reference

| Cell | What it does | Time |
|------|--------------|------|
| 1 | Check GPU | 5s |
| 2 | Install libraries | 1 min |
| 3 | Download 11k dataset | 5–10 min |
| 4 | Preview annotations | 10s |
| 5 | **Train YOLOv8s** | 60–90 min |
| 6 | View training curves | 10s |
| 7 | Get mAP score | 2 min |
| 8 | Test on your photo | 30s |
| 9 | **Full YOLO + OCR pipeline** | 30s |
| 10 | Save to Google Drive | 1 min |

## Common Errors

| Error | Fix |
|-------|-----|
| `CUDA out of memory` | Change `batch=16` to `batch=8` in Cell 5 |
| `FileNotFoundError best.pt` | Training didn't finish — re-run Cell 5 |
| `No labels found` | Re-run Cell 3 — paths get fixed automatically |
| Session disconnected | Re-run cells 2, 3 (data re-downloads), then 5 |
| OCR returns nothing | Lower `conf=0.25` to `conf=0.1` in Cell 9 |